In [ ]:
import numpy as np

K = 2
T = 10_000
M = 20

def sample_loss(rng, q):
    success = rng.binomial(n=1, p=q, size=1).item()
    return 1 if success else -1

In [ ]:
def eps_greedy(q1, q2, eps, rng):
    smallest = np.argmin([q1,q2])
    n = np.zeros(K)
    cumul_losses = np.zeros(K)
    epsT = round(eps*T)
    regret = 0

    for t in range(epsT+1):
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        I_t = np.argmin(n)
        n[I_t] += 1
        cumul_losses[I_t] += sampled_losses[I_t]
        regret += sampled_losses[I_t] - sampled_losses[smallest]
    best = np.argmin(cumul_losses)
    for t in range(epsT+1, T):
        regret += sampled_losses[best] - sampled_losses[smallest]
    
    return regret

In [ ]:
def lcb(q1, q2, delta, rng):
    smallest = np.argmin([q1,q2])
    n = np.repeat(1,K)
    regret = 0
    cumul_losses = np.array([sample_loss(rng, q1), sample_loss(rng, q2)])
    mu_hats = np.copy(cumul_losses)
    mu_deltas = mu_hats-np.sqrt(2*np.log(1/delta))

    for t in range(K, T):
        sampled_losses = [sample_loss(rng, q1), sample_loss(rng, q2)]
        I_t = np.argmin(mu_deltas)
        n[I_t] += 1
        cumul_losses[I_t] += sampled_losses[I_t]
        mu_hats = cumul_losses/n
        mu_deltas = mu_hats-np.sqrt(2*np.log(1/delta)/n)
        regret += sampled_losses[I_t] - sampled_losses[smallest]

    return regret